In [6]:
import torch

x = torch.tensor([2.0], requires_grad=True)   # 叶子节点
w = torch.tensor([3.0], requires_grad=True)   # 叶子节点
b = torch.tensor([1.0], requires_grad=True)   # 叶子节点

y = w * x + b   # y = 3 * 2 + 1 = 7
z = y.mean()    # 标量
z.backward()    # 自动求导

print(x.grad)   # dz/dx = w = 3.0
print(w.grad)   # dz/dw = x = 2.0
print(b.grad)   # dz/db = 1.0

print(z.grad)

tensor([3.])
tensor([2.])
tensor([1.])
None


/tmp/ipykernel_3183830/3086982697.py:15: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:489.)
  print(z.grad)


In [49]:
import torch
k = torch.randn(2, 3, 4)   # (B, S_k, d_k)
print(k.shape)
print(k)

k_T = k.transpose(-2,-1)   # (B, 64, 4)
print(k_T.shape)
print(k_T)

torch.Size([2, 3, 4])
tensor([[[ 0.1256,  0.0118, -2.1772, -0.7294],
         [-0.0843, -0.0891, -0.7083, -0.4594],
         [-1.2949,  0.6084,  0.7910,  0.5588]],

        [[ 0.1408, -0.4783, -1.6693,  0.2782],
         [ 0.1776, -0.8238, -0.4775, -0.6782],
         [-0.7122,  1.0932, -0.0635, -1.0823]]])
torch.Size([2, 4, 3])
tensor([[[ 0.1256, -0.0843, -1.2949],
         [ 0.0118, -0.0891,  0.6084],
         [-2.1772, -0.7083,  0.7910],
         [-0.7294, -0.4594,  0.5588]],

        [[ 0.1408,  0.1776, -0.7122],
         [-0.4783, -0.8238,  1.0932],
         [-1.6693, -0.4775, -0.0635],
         [ 0.2782, -0.6782, -1.0823]]])


In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F


class SingleHeadAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.qkv = nn.Linear(d_model, 3*d_model)  # 一次性生成 QKV
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        qkv = self.qkv(x).reshape(B, S, 3, D).permute(2,0,1,3)
        q, k, v = qkv[0], qkv[1], qkv[2]  # (B,S,D)
        scores = q @ k.transpose(-2,-1) / (D**0.5)
        if mask is not None:
            scores = scores.masked_fill(mask==0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        out = attn @ v
        return self.out(out)